In [15]:
import boto3
from boto3.dynamodb.conditions import Attr
from datetime import datetime
from uuid import uuid4

dynamodb = boto3.resource("dynamodb")

old_table = dynamodb.Table("time-tracker-entries")
new_table = dynamodb.Table("time-tracker-v2")

In [16]:
USER_ID = "roy"

CATEGORY_MAP = {
    "coursework": {
        "id": '5e2ce407-eb74-4d1d-abac-17a135edcbd6',
        "name": "Coursework"
    },
    "work": {
        "id": '019fb2df-ed45-4bd5-8de2-9bc84036a7f1',
        "name": "Work"
    },
    "prayer": {
        "id": '6fb5d7dd-bcfe-4692-b2dc-b374b2814bd7',
        "name": "Prayer"
    },
    "rest": {
        "id": 'a50435f6-7262-46ff-bc17-57d0a5df4b89',
        "name": "Rest"
    },
    "social": {
        "id": '89bb7646-c222-4587-b3de-2302cb919548',
        "name": "Social"
    },
    "family": {
        "id": '2fe0d15d-1180-4b21-9c9b-cc9fa48c2498',
        "name": "Family"
    },
    "self-study": {
        "id": '62a51329-1c18-4c61-9092-c666dd556471',
        "name": "Self Study"
    },
    "chores": {
        "id": 'd43db217-cdc8-4bf7-827c-90e58f002f68',
        "name": "Chores"
    },
    "exercise": {
        "id": '21dfd605-0c52-4abf-a3ee-d665eb1bac98',
        "name": "Exercise"
    }
}

def put_category(category_id, name):
    new_table.put_item(
        Item={
            "PK": f"USER#{USER_ID}",
            "SK": f"CATEGORY#{category_id}",
            "entityType": "Category",
            "categoryId": category_id,
            "name": name,
            "isActive": True,
            "createdAt": datetime.utcnow().isoformat() + "Z",
            "schemaVersion": 2,
        },
        ConditionExpression="attribute_not_exists(PK) AND attribute_not_exists(SK)"
    )


# 1. create categories
for cat in CATEGORY_MAP.values():
    put_category(cat["id"], cat["name"])

/var/folders/jm/rgjqkd2s5j5d309wnmpsvddr0000gn/T/ipykernel_63460/2470809839.py:51: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "createdAt": datetime.utcnow().isoformat() + "Z",


TokenRetrievalError: Error when retrieving token from sso: Token has expired and refresh failed

In [ ]:
{
 "day": "2026-02-16",
 "timestamp": "2026-02-16T00:20:32.877000+00:00",
 "category": "social",
 "id": "6cebde43-3855-4356-986a-9332b0147bfa"
}

In [23]:
from boto3.dynamodb.conditions import Key
import botocore.exceptions

def migrate_entry(old_item):
    old_category = old_item["category"]
    category = CATEGORY_MAP[old_category]

    id = old_item.get("id")
    timestamp = old_item["timestamp"]

    new_item = {
        "PK": f"USER#{USER_ID}",
        "SK": f"ENTRY#{timestamp}",
        "entityType": "TimeEntry",
        "id": id,
        "categoryId": category["id"],
        "categoryNameSnapshot": category["name"],
        "timestamp": old_item["timestamp"],
        "schemaVersion": 2,
        "migratedFrom": {
            "day": old_item["day"],
            "timestamp": old_item["timestamp"],
            "category": old_item["category"],
            "id": old_item["id"],
        },
    }

    new_table.put_item(
        Item=new_item,
        ConditionExpression="attribute_not_exists(PK) AND attribute_not_exists(SK)",
    )
# 2. scan and migrate entries
response = old_table.query(KeyConditionExpression=Key("day").eq("2026-05-25"), ScanIndexForward=True)
items = response["Items"]

# while "LastEvaluatedKey" in response:
#     response = old_table.scan(ExclusiveStartKey=response["LastEvaluatedKey"])
#     items.extend(response["Items"])

for item in items:
    if "category" in item:
        try:
            migrate_entry(item)
            print("migrated")
        except botocore.exceptions.ClientError as e:
            print(e.response['Error']['Code'])
